# Тензоры, формы и устройства

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Создавать тензоры на CPU и согласовывать формы перед вычислением ошибок.

В NumPy мы уже работали с формами и матричными действиями. PyTorch добавит возможность автоматически находить производные по выбранным числам — будущим весам модели. Начнём со знакомых вычислений над массивами; в следующем уроке свяжем их с градиентом. Векторы, матрицы и средняя квадратичная ошибка сохраняют прежний математический смысл.

## Тензоры, типы и устройства

CPU — центральный процессор компьютера. В этом уроке все данные и вычисления размещены на нём; видеокарта не нужна.

Тензор здесь — многомерный массив чисел. `torch.tensor` создаёт его из чисел или вложенных списков. В PyTorch он имеет форму `shape`, тип `dtype` и устройство `device`. Числовой таблице обычно подходит `torch.float32`; для проверки производных с меньшей погрешностью округления удобно взять `float64`. Ни один из этих типов не хранит произвольные дроби совершенно точно. Для целых индексов часто нужен `torch.int64`. Параметр `requires_grad=True` указывает, что по этому тензору понадобятся производные. В примере `w` — два веса линейной модели, `X` — два объекта, `y` — их правильные ответы. Оператор `@` умножает матрицу на вектор.

In [ ]:
import torch

X = torch.tensor([[2., 1.], [1., 3.]], dtype=torch.float64, device="cpu")
w = torch.tensor([1., 1.], dtype=torch.float64, requires_grad=True)
y = torch.tensor([5., 4.], dtype=torch.float64)
assert X.shape == (2, 2) and w.shape == (2,)
assert X.device.type == "cpu"
prediction = X @ w
assert prediction.shape == y.shape


В [математическом выводе](../../../math/calculus/partial-derivatives/) w и y записаны столбцами. Здесь `w.shape == (2,)` и `y.shape == (2,)`: PyTorch, как и NumPy, позволяет хранить векторы с одной осью. Произведение `(n, d) @ (d,)` даёт `(n,)`; со столбцом `(d, 1)` результат имел бы форму `(n, 1)`. Числа описывают тот же прогноз, но перед вычитанием формы нужно согласовать.

Broadcasting работает по знакомым правилам NumPy: формы ответов и предсказаний проверяйте до вычитания. `.to(device)` переносит тензор и возвращает результат; все участники операции должны быть на совместимых устройствах. В наших примерах ускоритель не нужен.

## Разобранный пример: сохраняем одну ошибку на объект

У модели с одним числовым ответом предсказание часто имеет форму `(n, 1)`, а ответы — `(n,)`. Перед вычитанием приведём обе стороны к одной форме. Метод `reshape` задаёт нужную форму, сохраняя число элементов, а `torch.allclose` сравнивает тензоры с допуском.

In [ ]:
predicted_column = torch.tensor([[3.], [4.]], dtype=torch.float64)
predicted_vector = predicted_column.reshape(2)
assert predicted_vector.shape == y.shape
errors = predicted_vector - y
assert torch.allclose(errors, torch.tensor([-2., 0.], dtype=torch.float64))


Если вычесть столбец и вектор без проверки, broadcasting создаст таблицу 2 × 2. Вычисление не обязательно упадёт с ошибкой: неверным окажется смысл результата.

## Тип — часть входных данных

Числовые измерения обычно хранят дробным типом, индексы — целым. Устройство и тип должны быть согласованы у участников операции. `.to(device="cpu", dtype=...)` возвращает преобразованный тензор; результат нужно сохранить. Выбор устройства не меняет математическую форму задачи.

## Проверяем численный смысл произведения

При весах `[1, 1]` первый объект даёт `2 * 1 + 1 * 1 = 3`, второй — `1 * 1 + 3 * 1 = 4`. Поэтому предсказания равны `[3, 4]`, а ошибки относительно `[5, 4]` равны `[-2, 0]`. Форма проверяет количество ответов; численный пример проверяет, что мы умножили нужные величины.

## Задачи

### 1. Базовая

Что означает shape (4, 3)?

<details>
<summary>Подсказка</summary>

Первая ось — объекты.

</details>



### 2. Базовая

Нужна ли видеокарта для примеров урока?

<details>
<summary>Подсказка</summary>

Используется device cpu.

</details>



### 3. Прикладная

Создайте тензор [[1, 2], [3, 4]] типа float64.

<details>
<summary>Подсказка</summary>

Укажите dtype.

</details>



### 4. Прикладная

Найдите произведение этой матрицы на вектор [1, 1].

<details>
<summary>Подсказка</summary>

Используйте @.

</details>



### 5. Олимпиадная

Какова форма разности (4, 1) и (4,)?

<details>
<summary>Подсказка</summary>

Согласуйте размеры справа налево.

</details>



### 6. Олимпиадная

Почему успешное выполнение операции ещё не подтверждает правильность форм?

<details>
<summary>Подсказка</summary>

Broadcasting допускает многие сочетания.

</details>



## Итоги

Вы умеете создавать CPU-тензоры нужного типа и проверять форму результата.

## Самопроверка

Для матрицы `(4, 3)` и весов `(3,)` назовите форму произведения. Почему `(4, 1)` нужно привести к `(4,)` перед сравнением с вектором ответов?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Четыре строки по три элемента.

### Задача 2

Нет.

### Задача 3

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]], dtype=torch.float64)
assert a.shape == (2, 2)


### Задача 4

In [ ]:
answer = a @ torch.tensor([1., 1.], dtype=torch.float64)
assert torch.allclose(answer, torch.tensor([3., 7.], dtype=torch.float64))


### Задача 5

(4, 4). Это попарные разности, а не четыре ошибки.

### Задача 6

Совместимая операция может считать другую величину. Нужно заранее выписать ожидаемую форму и проверить её.